# 03 · Multi-LoRA — un Gemma, muchos Gemmas**Charla 2, slides 22–23. El clímax.**---## La ideaEl base **nunca se toca**. Un adapter pesa 60 MB.Entonces: **¿por qué tendrías un solo modelo?**```                    ┌─ adapter: extracción de labs      (60 MB)Gemma 4 E4B  ───────┼─ adapter: notas periodontales     (60 MB)(base, 1 vez)       ├─ adapter: QA sobre FHIR           (60 MB)   ~4 GB            └─ adapter: dictado clínico         (60 MB)```Un router elige el adapter **por request**.## Los números que lo hacen viable| | ||---|---|| Overhead de swap por request (vLLM) | **sub-milisegundo** || Adapters concurrentes en una A100 | **~200**, con <5% de latencia extra || Cómputo extra por capa a r=16 | **<1%** (<7% a r=64) || Carga | **on-demand desde disco/S3, evicción LRU** |## Las dos restricciones1. Todos los adapters comparten **arquitectura base y dtype**. No mezclás bases.2. **Esto es serving en GPU. NO aplica al Gallery en el celular.** En el borde,   el mecanismo de especialización **no es el adapter — es la skill**.Y ahí se cierra el círculo de las dos charlas: **adapter en la GPU, skill en el borde.**---⚠️ **Este notebook necesita más VRAM que una T4.** Corré con A100 (Colab Pro) oadaptá a un solo adapter para demostrar el mecanismo con menos memoria.

In [ ]:
!pip install -q vllm!nvidia-smi --query-gpu=name,memory.total --format=csv    

## 1. Entrenar (o traer) varios adaptersEn la charla usamos el `adapter_labs` del notebook 02. Para la demo completaharían falta 2–4 adapters distintos, cada uno entrenado con el mismo base.Acá asumimos que ya existen en disco. Cada uno debe tener:- `adapter_config.json`- `adapter_model.safetensors`Ese par es lo único que vLLM necesita.

In [ ]:
import osADAPTERS = {    "labs":         "/content/adapter_labs",          # del notebook 02    "periodontal":  "/content/adapter_periodontal",   # entrenar con el mismo base    "fhir":         "/content/adapter_fhir",}for nombre, ruta in ADAPTERS.items():    existe = os.path.isdir(ruta)    print(f"{'✓' if existe else '✗'} {nombre:14s} {ruta}")    if existe:        print(f"     {os.listdir(ruta)}")    

## 2. Servir: UN base, N adapters`enable_lora=True` es todo. `max_loras` es cuántos pueden estar activos a la vez;`max_lora_rank` tiene que ser ≥ el rango con el que entrenaste (usamos 16).

In [ ]:
from vllm import LLM, SamplingParamsfrom vllm.lora.request import LoRARequestBASE = "google/gemma-4-E4B-it"   # <-- el MISMO base con el que entrenaste los adaptersllm = LLM(    model = BASE,    enable_lora = True,    max_loras = 4,          # activos en simultáneo    max_lora_rank = 16,     # >= el r que usaste al entrenar    max_model_len = 4096,    dtype = "bfloat16",    gpu_memory_utilization = 0.90,)sampling = SamplingParams(temperature=0.0, max_tokens=256)    

## 3. El routerAcá está el punto: **el mismo `llm`, la misma GPU, la misma memoria** — y tresespecialistas distintos. Lo único que cambia es el `LoRARequest`.En un sistema real, la elección del adapter la hace el agente (o un clasificadorchico, o una regla). Acá lo hacemos explícito para que se vea el mecanismo.

In [ ]:
# El id numérico tiene que ser único y estable por adapter.LORA_REQS = {    nombre: LoRARequest(nombre, i + 1, ruta)    for i, (nombre, ruta) in enumerate(ADAPTERS.items())    if os.path.isdir(ruta)}def responder(prompt: str, especialista: str | None = None) -> str:    """Si especialista es None -> el BASE pelado. Si no -> base + adapter."""    lora = LORA_REQS.get(especialista) if especialista else None    out = llm.generate([prompt], sampling, lora_request=lora)    return out[0].outputs[0].text.strip()informe = "LABORATORIO CENTRAL\nGlucosa: 187.4 mg/dL  (VR 70-110)"print("── BASE PELADO ────────────────────────────────")print(responder(informe))print("\n── + adapter 'labs' ───────────────────────────")print(responder(informe, "labs"))    

### Lo que hay que decir cuando corre esa celda> "Miren la diferencia. **El base es el mismo. La GPU es la misma. La memoria es la misma.**> Lo único que cambió son sesenta megas.>> Eso es un especialista médico entero, que pesa menos que una foto."

## 4. Batch con adapters mezcladosEsto es lo que hace que la arquitectura sea viable en producción: vLLM **batchearequests que usan adapters distintos**, en la misma pasada.

In [ ]:
pedidos = [    ("LABORATORIO CENTRAL\nCreatinina: 2.8 mg/dL  (VR 0.6-1.3)", "labs"),    ("Sondaje 3-4-3 en 16, sangrado al sondaje positivo, movilidad grado I.", "periodontal"),    ("¿Qué recurso FHIR representa un resultado de laboratorio?", "fhir"),    ("Resumí en una oración qué es la periodontitis.", None),   # base pelado]for prompt, esp in pedidos:    etiqueta = esp or "BASE"    if esp and esp not in LORA_REQS:        print(f"[{etiqueta:12s}] (adapter no encontrado, salteado)")        continue    print(f"\n[{etiqueta:12s}] {prompt[:50]}...")    print(f"             -> {responder(prompt, esp)[:200]}")    

## 5. Servidor OpenAI-compatible (lo que usarías de verdad)Fuera del notebook, esto se levanta así:```bashvllm serve google/gemma-4-E4B-it \  --enable-lora \  --max-loras 4 \  --max-lora-rank 16 \  --lora-modules \      labs=/models/adapter_labs \      periodontal=/models/adapter_periodontal \      fhir=/models/adapter_fhir```Y el cliente elige el especialista **con el campo `model`**:```pythonfrom openai import OpenAIclient = OpenAI(base_url="http://localhost:8000/v1", api_key="-")client.chat.completions.create(    model="labs",              # <-- ACÁ se elige el adapter    messages=[{"role": "user", "content": informe}],    extra_body={"chat_template_kwargs": {"enable_thinking": True}},    #             ^^^^ en vLLM esto prende razonamiento Y function calling.    #                  Si tu agente no llama tools, es por esto.)````--lora-modules` también acepta rutas remotas (S3): los adapters se cargan al primerrequest y se desalojan de un caché LRU. No necesitás tenerlos todos en memoria.---## CierreLo que acabás de ver es **un sistema multi-agente con varios especialistas médicoscorriendo en una sola placa de video**.No es un truco de pizarrón. Es la arquitectura de un producto real que tiene quecumplir HIPAA.**Y lo podés construir este fin de semana.**